# M0 offline smoke graph

This notebook delegates preparation to the public CLI handler; it does not duplicate parsing or graph-building code.

In [ ]:
from pathlib import Path
import json
import tempfile
from semmap_haken.cli import main
from semmap_haken.manifest import RunManifest
from semmap_haken.notebook import detect_environment, make_execution_metadata, persist_paths, resolve_notebook_paths, run_resource_preflight

ALLOW_PRODUCTION_DOWNLOAD = False
USE_DRIVE = False
CONFIG_TEMPLATE = Path('configs/conceptnet_en_smoke.yaml')
FIXTURE = Path('tests/fixtures/conceptnet_tiny.tsv').resolve()
paths = resolve_notebook_paths(CONFIG_TEMPLATE)
environment = detect_environment()
preflight = run_resource_preflight(Path('configs/resource_profiles/smoke.yaml'), paths.data_root)
print({'fixture': str(FIXTURE), 'exists': FIXTURE.is_file(), 'preflight_ok': preflight.ok, 'runs_root': str(paths.runs_root)})

In [ ]:
# The temporary config selects the offline fixture. No production download occurs in default execution.
with tempfile.TemporaryDirectory() as temporary:
    config_path = Path(temporary) / 'offline_smoke.yaml'
    config_path.write_text(f'''
paths: {{workspace_root: {paths.workspace_root}, data_root: data, cache_root: data/cache, runs_root: runs}}
dataset: {{source: offline-tiny, path: {FIXTURE}, language: en, relations: [RelatedTo, IsA], min_weight: 1.0, max_nodes: 1000, component: largest}}
graph: {{directed: false, weight_transform: log1p, operator: normalized_adjacency}}
runtime: {{profile: smoke, random_seed: 1729, resource_profile: {Path('configs/resource_profiles/smoke.yaml').resolve()}}}
''', encoding='utf-8')
    assert main(['prepare', '--config', str(config_path)]) == 0
run_dirs = sorted(paths.runs_root.glob('prepare-*'), key=lambda item: item.stat().st_mtime)
run_dir = run_dirs[-1]
manifest = RunManifest.read_json(run_dir / 'manifest.json')
print(json.dumps({'run_id': manifest.run_id, 'artifact_paths': manifest.artifacts, 'stages': manifest.stages}, indent=2))

In [ ]:
# Optional persistence copies only selected light metadata after active-local execution.
PERSIST_ROOT = None
records = [] if PERSIST_ROOT is None else persist_paths([run_dir / 'manifest.json', run_dir / 'graph_metadata.json'], source_root=paths.runs_root, destination_root=PERSIST_ROOT)
metadata = make_execution_metadata(notebook_name='01_data_smoke_and_sparse_graph.ipynb', environment=environment, paths=paths, drive_enabled=USE_DRIVE, persisted_paths=[record.destination for record in records])
metadata